# Lesson 4 : Hosted Tools

Agent Framework provides a wide variety of built-in tool's object (such as, Code Interpreter, Web Search, File Search, MCP tools, etc), and you can add these tools in your agent.

> Note : For available built-in tools, see [API reference](https://learn.microsoft.com/en-us/python/api/agent-framework-core/agent_framework).

The hosted tools in Agent Framework are built-in tools hosted on individual client.  
For instance, ```HostedWebSearchTool``` will seach web to get information. This object is implemented as [OpenAI's web search tool](https://platform.openai.com/docs/guides/tools-web-search), when the client object is ```OpenAIResponsesClient``` or ```AzureAIClient```. It'll use [Bing grounding tool](https://learn.microsoft.com/en-us/python/api/azure-ai-agents/azure.ai.agents.models.binggroundingtool) or [Bing custom search tool](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/tools-classic/bing-custom-search-samples), when ```AzureAIAgentClient``` (which runs on Azure AI agents). And, it'll use [Claude's web search tool](https://platform.claude.com/docs/en/agents-and-tools/tool-use/web-search-tool), when ```AnthropicClient```.

In this exercise, we explorer hosted tools, including MCP tool's calling.

## HostedWebSearchTool

First, we change the example in Lesson 1 to use ```HostedWebSearchTool```.

Same as in Lesson 1, we create a ```AzureAIClient``` object as follows.  
As I have mentioned above, this object uses OpenAI's web search tool as ```HostedWebSearchTool```.

In [1]:
from agent_framework.azure import AzureAIClient
from azure.identity.aio import AzureCliCredential

credential = AzureCliCredential()
client = AzureAIClient(
    credential=credential,
)

Next we build an agent which uses ```HostedWebSearchTool``` as follows, instead of local function's tools.

In [2]:
from agent_framework import HostedWebSearchTool

agent = client.create_agent(
    name="WeatherAgentWithSearchTool",
    instructions="あなたは、気象情報に関する Agent です。",  # "you are an agent about weather information."
    tools=[HostedWebSearchTool()],
)

Now we run the agent.  
As you see, this agent knows "what day is it today" or "what is the weather condition today".

In order to get information, the agent will first invoke OpenAI's built-in web search tool. (Azure OpenAI Responses API will handle this process internally.)

In [3]:
from IPython.display import Markdown, display

result = await agent.run("今日の大阪の天気と気温を教えてください。")  # "tell me the weather and temperature in Osaka today."
display(Markdown(result.text))

今日（**2026年1月13日**）の大阪は、**くもりがちで、午後に弱い雨（霧雨・小雨）が降る可能性**があります。

- **気温**：**最高 約12℃／最低 約3℃**（目安）

必要なら、「大阪市内（梅田あたり）」など地点を指定して、もう少しピンポイントの予報（時間帯別）でもまとめます。[Osaka Weather Forecast](https://www.weather-forecast.com/locations/Osaka/forecasts/latest)[Weather Osaka - meteoblue](https://www.meteoblue.com/en/weather/week/osaka_japan_1853904)[Weather - Osaka City - 14-Day Forecast & Rain | Ventusky](https://www.ventusky.com/osaka)[Weather for Osaka, Japan - timeanddate.com](https://www.timeanddate.com/weather/japan/osaka)[Osaka-shi, Japan Weather Conditions | Weather Underground](https://www.wunderground.com/weather/jp/osaka-shi)

## HostedMCPTool

```HostedMCPTool``` uses MCP tool calling hosted on the client.  
In this example (```AzureAIClient``` object), MCP server integration in Azure OpenAI Responses API (see [here](https://platform.openai.com/docs/guides/tools-connectors-mcp)) will be used internally.

> Note : On contrary, using built-in ```MCPStdioTool```, ```MCPStreamableHTTPTool```, and ```MCPWebsocketTool```, MCP is handled by the implementation in Agent Framework SDK. (It calls MCP servers to list available tools, and each tool's calling is handled as a local function in Agent Framework SDK.)  
> This might be useful when MCP (or some part of MCP specification) is not supported in your client (e.g., some remote API models might not support MCP Stdio server tool), because this doesn't depend on the client's capability. (Compared to ```HostedMCPTool```, however, this incurs overhead in your computing environment.)

In this example, we create an agent to answer Microsoft technical questions.  
This agent uses a remote MCP server (Streamable HTTP server), which provides information about Microsoft Learn document.

In [4]:
from agent_framework import HostedMCPTool

agent = client.create_agent(
    name="MSTechKnowledgeAgent",
    instructions="あなたは、Microsoft の製品やサービスに関する技術質問に答える Agent です。",  # "you are an agent who answers technical questions about Microsoft products and services."
    tools=[HostedMCPTool(
        name="Microsoft Learn MCP",
        url="https://learn.microsoft.com/api/mcp",
        approval_mode="never_require",
    )],
)

Let's ask a technical question about Microsoft Azure.  
This call handles MCP tool calling (about Microsoft Learn document) in Azure OpenAI Responses API.

In [5]:
from IPython.display import Markdown, display

result = await agent.run("Azure CLI を使って Azure Storage アカウントを作成する方法を教えてください。")  # "how to create an Azure storage account using Azure CLI ?"
display(Markdown(result.text))

Azure CLI で Storage アカウント（一般用途 v2: StorageV2）を作成する基本手順は「(1) サインイン → (2) リソース グループ作成 → (3) ストレージ アカウント作成」です。

## 1) サインイン（ローカル実行の場合）
```bash
az login
```

## 2) リソース グループを作成
```bash
az group create \
  --name storage-resource-group \
  --location eastus
```

## 3) Storage アカウントを作成
※ Storage アカウント名（`--name`）は Azure 全体で一意（3～24 文字・英小文字と数字のみ）にする必要があります。

```bash
az storage account create \
  --name <account-name> \
  --resource-group storage-resource-group \
  --location eastus \
  --sku Standard_RAGRS \
  --kind StorageV2 \
  --min-tls-version TLS1_2 \
  --allow-blob-public-access false
```

## 参考（公式ドキュメント）
- Create an Azure storage account: https://learn.microsoft.com/azure/storage/common/storage-account-create#create-a-storage-account